Our project will simulate how fast *E. coli* grows under normal conditions, and then we will "play God" by altering its environment (suffocating it) and mutating its genetics (deleting a gene) to see how the mathematics predict its survival.

To do this, we will use **Python** and a famous scientific library called **COBRApy** (COnstraint-Based Reconstruction and Analysis). 

### Phase 1: Setting up the Virtual Laboratory

If you are running this yourself, first install the library:

In [1]:
pip install cobra

  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.6 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 773.7 kB/s eta 0:00:001m753.6 kB/s eta 0:00:01
Using cached appdirs-1.4.4-py2.py3-none-any.whl (9.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━

Now, let's load our virtual organism. We will use the **"E. coli Core Model"**. This is a simplified blueprint of *E. coli*’s central metabolism containing 95 reactions and 72 metabolites—perfect for learning.

In [2]:
import cobra

# 1. Load the core E. coli blueprint
model = cobra.io.load_model("textbook")

print(f"Number of Metabolites: {len(model.metabolites)}")
print(f"Number of Reactions: {len(model.reactions)}")
print(f"Objective (Goal): {model.objective.expression}")

Number of Metabolites: 72
Number of Reactions: 95
Objective (Goal): 1.0*Biomass_Ecoli_core - 1.0*Biomass_Ecoli_core_reverse_2cdba


**Expected Output:**
> Number of Metabolites: 72
> Number of Reactions: 95
> Objective (Goal): 1.0 * Biomass_Ecoli_core - 1.0 * Biomass_Ecoli_core_reverse

### Phase 2: Experiment 1 - The "All-You-Can-Eat" Buffet (Aerobic Growth)

By default, the core model assumes the bacterium is in a warm petri dish with plenty of **glucose** and **oxygen** (Aerobic conditions). Let's run Flux Balance Analysis (FBA) to find the maximum possible growth rate.

In [3]:
# Run Flux Balance Analysis (FBA)
solution = model.optimize()

# Print the predicted growth rate (Biomass flux)
print(f"Aerobic Growth Rate: {solution.objective_value:.3f} per hour")

# Let's check how much Glucose and Oxygen the cell is consuming
print(f"Glucose consumed: {solution.fluxes['EX_glc__D_e']:.2f}")
print(f"Oxygen consumed: {solution.fluxes['EX_o2_e']:.2f}")

Aerobic Growth Rate: 0.874 per hour
Glucose consumed: -10.00
Oxygen consumed: -21.80


**Expected Output:**
> Aerobic Growth Rate: 0.874 per hour
> Glucose consumed: -10.00
> Oxygen consumed: -21.80

*(Note: Consumption fluxes are negative because they are leaving the environment and entering the cell).*
**What this means:** Under perfect conditions, our *E. coli* mathematically splits and doubles roughly every 45-50 minutes, consuming 10 units of glucose and 21.8 units of oxygen.


### Phase 3: Experiment 2 - Suffocating the Cell (Anaerobic Growth)

What happens if we seal the petri dish so no oxygen gets in? We need to change the **constraints** of our mathematical model. We will set the maximum oxygen uptake limit to zero.

In [4]:
# Change the lower bound of Oxygen exchange to 0 (No oxygen enters)
model.reactions.EX_o2_e.lower_bound = 0

# Re-run FBA with the new constraints
anaerobic_solution = model.optimize()

print(f"Anaerobic Growth Rate: {anaerobic_solution.objective_value:.3f} per hour")

# Let's see if the cell is producing a new byproduct to survive
print(f"Ethanol produced: {anaerobic_solution.fluxes['EX_etoh_e']:.2f}")
print(f"Acetate produced: {anaerobic_solution.fluxes['EX_ac_e']:.2f}")

Anaerobic Growth Rate: 0.212 per hour
Ethanol produced: 8.28
Acetate produced: 8.50


**Expected Output:**
> Anaerobic Growth Rate: 0.212 per hour
> Ethanol produced: 8.28
> Acetate produced: 8.58

**What this means:** This is the magic of FBA! Because oxygen is gone, the computer realizes the cell can no longer use cellular respiration (which makes a ton of ATP). To satisfy the math ($S \times v = 0$), the cell *must* route the flux through fermentation instead. The growth rate drops dramatically (from 0.874 to 0.212), and the computer correctly predicts that the cell will start secreting **ethanol and acetate** (acids/alcohol) as waste!

### Phase 4: Experiment 3 - The Genetic Knockout & The Metabolic Whirlpool
Now, let's play genetic engineer. We will delete a vital glycolysis gene that produces the enzyme *Phosphofructokinase (PFK)*. 

To simulate this accurately, we will use an advanced tool called **Parsimonious FBA (pFBA)**. Normal FBA only maximizes growth rate, which can sometimes result in impossible math glitches. pFBA does a two-step optimization: First, it maximizes the growth rate. Second, it finds the route that requires the **absolute minimum chemical effort** (minimizing the sum of all internal fluxes) to achieve that growth. Cells are naturally efficient, so pFBA acts closer to real biology.

In [11]:
# Import the specialized pFBA tool
from cobra.flux_analysis import pfba

# 1. Reset the model to fresh (ACTUALLY turn the oxygen back on!)
model.reactions.EX_o2_e.lower_bound = -1000.0

with model:
    # 2. Find the PFK reaction and force its flux to 0 (simulate a knockout)
    model.reactions.PFK.knock_out()
    
    # 3. Run Parsimonious FBA (pFBA) on the mutant
    parsimonious_solution = pfba(model)
    
    # In pFBA, the mathematical objective becomes the "Sum of all fluxes"
    # To get the actual growth rate, we check the Biomass flux directly:
    print(f"Mutant Growth Rate: {parsimonious_solution.fluxes['Biomass_Ecoli_core']:.3f} per hour")
    print(f"Total Chemical Effort (Sum of Fluxes): {parsimonious_solution.objective_value:.3f}")
    
    # Let's check how the cell is bypassing the broken enzyme
    print(f"Flux through normal pathway (PFK): {parsimonious_solution.fluxes['PFK']:.2f}")
    print(f"Flux through alternate pathway (G6PDH2r): {parsimonious_solution.fluxes['G6PDH2r']:.2f}")

Mutant Growth Rate: 0.704 per hour
Total Chemical Effort (Sum of Fluxes): 680.110
Flux through normal pathway (PFK): 0.00
Flux through alternate pathway (G6PDH2r): 27.90


**Expected Output:**
> Mutant Growth Rate: 0.704 per hour
> Total Chemical Effort (Sum of Fluxes): 680.110
> Flux through normal pathway (PFK): 0.00
> Flux through alternate pathway (G6PDH2r): 27.90

#### The Biological Explanation: Stoichiometric Trapping
When you read the output, two numbers stand out as very strange:
1. The objective value is **680.110**. As explained, in pFBA, the solver changes its goal from "Maximize Growth" to "Minimize total effort." The mutant cell's total sum of fluxes required to survive is 680.110, while its actual growth rate remains 0.704.
2. The flux through the alternate pathway is **27.90**. *How can a cell eating only 10 units of sugar process 27.90 units through a single enzyme?* 

This reveals a fascinating phenomenon called **Stoichiometric Trapping**:
Because the PFK enzyme is dead, a sugar intermediate called *Fructose-6-Phosphate (F6P)* cannot move forward through its normal metabolic door. It is trapped. To survive, the cell diverts the incoming 10 units of glucose into an alternate detour (the Pentose Phosphate Pathway, starting with **G6PDH2r**). 

However, part of this alternate pathway spits out F6P. Because the forward door is still bricked shut, the F6P will build up and kill the cell. To avoid this, the cell frantically runs another enzyme in reverse, turning the trapped F6P *back* into starting sugar, and shoving it right back into **G6PDH2r** again! 

To process the 10 units of food coming from the environment, this pathway has to operate like a metabolic whirlpool, recycling the trapped sugars and spinning nearly three times as fast (roughly 3 x 10 = 30) just to keep the lights on. 


### Project Summary

Using simple matrix algebra and constraints (FBA), our tiny project just successfully predicted:
1. The baseline growth rate of *E. coli*.
2. The exact metabolic shift from respiration to fermentation (producing alcohol) when oxygen is removed.
3. The alternate biological pathways the cell will use to survive a genetic mutation.

This exact methodology is used today by pharmaceutical companies to engineer yeast that produce insulin, and by green-tech companies to engineer bacteria that eat plastic or produce biofuels! 